# Lecture 7 — Convolutional and Residual Networks
## Lab Notebook · Deep Learning · UCU

In Notebook 5, we trained a deep MLP on tabular data (UCI Covertype) and learned how to *measure* whether it generalizes — train/val/test discipline, regularization, hyperparameter search. In this notebook, we move to **images**, where MLPs hit a hard ceiling and convolutional layers replace the dense weight matrix with a small **shared kernel**.

We will work with the **Oxford-IIIT Pet** dataset and use it three different ways:

1. **Part 1** — Build a small CNN *from scratch* for binary cat-vs-dog classification.
2. **Part 2** — Fine-tune a *pretrained* ResNet18 for 37-breed classification.
3. **Part 3** — Build and train a small *U-Net* for foreground/background segmentation.

The same pixels feed three networks with three different targets. By the end you will have an end-to-end mental model for the three computer-vision workflows that dominate industry.

### By the end of this notebook, you will:

1. Build a small CNN from scratch with `nn.Conv2d`, `nn.BatchNorm2d`, `nn.MaxPool2d`, and an `nn.AdaptiveAvgPool2d` head, and reason about its receptive field and parameter count.
2. Fine-tune a pretrained `resnet18` from `torchvision.models` on a new classification task by freezing the backbone and replacing the final layer.
3. Build a small **U-Net** for foreground/background segmentation, including encoder/decoder blocks and **skip connections**, and train it with `CrossEntropyLoss` on a per-pixel target.
4. Visualize segmentation predictions and qualitatively probe the model's generalization on an out-of-distribution image.

### Useful references

| Topic | Link |
|---|---|
| `nn.Conv2d`, `nn.BatchNorm2d`, `nn.MaxPool2d`, `nn.AdaptiveAvgPool2d`, `nn.ConvTranspose2d` | https://pytorch.org/docs/stable/nn.html |
| Pretrained models | https://pytorch.org/vision/stable/models.html |
| `OxfordIIITPet` dataset | https://pytorch.org/vision/stable/generated/torchvision.datasets.OxfordIIITPet.html |
| U-Net (Ronneberger et al., 2015) | https://arxiv.org/abs/1505.04597 |
| Lecture 7 notes | `../../lectures/lecture 7/notes.md` |
| Prince — *Understanding Deep Learning* | Ch. 10–11 |


---
## Setup

In [ ]:
# Colab setup (no-op locally)
import sys
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    !pip install -q torch torchvision torchinfo


In [ ]:
import os, time, warnings
warnings.filterwarnings('ignore')
from io import BytesIO
import urllib.request

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from torchvision.datasets import OxfordIIITPet
from torchvision import models
from PIL import Image
import matplotlib.pyplot as plt

torch.manual_seed(42); np.random.seed(42)

device = torch.device(
    'cuda' if torch.cuda.is_available()
    else ('mps' if torch.backends.mps.is_available() else 'cpu')
)
print(f'Device: {device}')
print(f'PyTorch: {torch.__version__}')

# UCU color palette
C1, C2, C3 = '#19326E', '#50ACB0', '#CD742A'
C4, C5, C6, C7 = '#A3477F', '#907FAB', '#4294CC', '#89A943'

if device.type == 'cpu':
    print('\n[!] No GPU detected. Part 2 (ResNet at 224x224) and Part 3 (U-Net) will be slow on CPU.')


### Download the dataset (~800 MB, run once)

We use **Oxford-IIIT Pet**: 7,349 photos of 37 breeds (cats and dogs), each annotated with the breed *and* a per-pixel **trimap** (foreground / background / boundary). We will load it three different ways via `target_types`:

- `'binary-category'` → `0` = cat, `1` = dog (Part 1)
- `'category'` → `0`…`36`, one per breed (Part 2)
- `'segmentation'` → trimap PIL image (Part 3)


In [ ]:
DATA_ROOT = './data'
os.makedirs(DATA_ROOT, exist_ok=True)

# Trigger download once with download=True
_ = OxfordIIITPet(root=DATA_ROOT, split='trainval', target_types='category', download=True)
_ = OxfordIIITPet(root=DATA_ROOT, split='test',     target_types='category', download=True)
print('Dataset downloaded.')


In [ ]:
# ImageNet normalization stats (used by all three parts)
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]


> **Practical callout — ImageNet normalization is the norm.**
> When using `torchvision` pretrained weights, always preprocess with the ImageNet mean/std the model was trained on. Even when training from scratch on natural photos, those values are sane defaults — they roughly center each channel on zero with unit variance.


---
## Part 1 — Build a CNN from scratch (~25 min)

The lecture's MNIST-1D experiment showed that a 2,050-parameter conv net beats a 150,000-parameter MLP on the same task. Convolution is best understood as a **strong inductive bias** that matches the symmetries of image data.

We will build a small VGG-style CNN — three `Conv → BN → ReLU → MaxPool` blocks plus a head — and train it to classify cats vs. dogs at 64×64.


### Exercise 1.1 — Build the data loaders

Load `OxfordIIITPet` with `target_types='binary-category'` for both splits at **64×64** with ImageNet normalization. The training transform should include a `RandomHorizontalFlip` (it is a free 2× data-augmentation since pet photos are not orientation-sensitive); the eval transform should not. Do not forget to include image normalization, using `IMAGENET_MEAN` and `IMAGENET_STD`. Use `batch_size=64` for train and 128 for test, `shuffle=True` for train only, `num_workers=2`.

In [ ]:
SIZE_P1 = 64

# YOUR CODE HERE: tf_p1_train, tf_p1_eval (use IMAGENET_MEAN / IMAGENET_STD)
raise NotImplementedError()

train_p1 = OxfordIIITPet(DATA_ROOT, split='trainval', target_types='binary-category', transform=tf_p1_train)
test_p1  = OxfordIIITPet(DATA_ROOT, split='test',     target_types='binary-category', transform=tf_p1_eval)
print(f'Train: {len(train_p1)}   Test: {len(test_p1)}')

img, y = train_p1[0]
print(f'Image shape: {tuple(img.shape)}   Label: {y}  (0=cat, 1=dog)')

# YOUR CODE HERE: train_loader_p1, test_loader_p1
raise NotImplementedError()


Quick visual check — plot 8 random images and their labels.

In [ ]:
# Visualize a small batch (uses an unnormalized loader for display only)
display_tf = transforms.Compose([transforms.Resize((SIZE_P1, SIZE_P1)), transforms.ToTensor()])
display_ds = OxfordIIITPet(DATA_ROOT, split='trainval', target_types='binary-category', transform=display_tf)

fig, axes = plt.subplots(2, 4, figsize=(10, 5))
idxs = np.random.choice(len(display_ds), 8, replace=False)
for ax, k in zip(axes.flat, idxs):
    img, y = display_ds[k]
    ax.imshow(img.permute(1, 2, 0)); ax.set_title('cat' if y == 0 else 'dog'); ax.axis('off')
plt.tight_layout(); plt.show()


### Exercise 1.2 — Build `SmallCNN`

Three `Conv → BatchNorm → ReLU → MaxPool` blocks (channel widths `3 → 16 → 32 → 64`) followed by an `AdaptiveAvgPool2d(1) → Flatten → Linear(64, num_classes)` head. Use `kernel_size=3` and `padding=1` so spatial resolution halves *only* at each pool. Each pool halves: 64 → 32 → 16 → 8.

> **Practical callout — no conv bias before BN.** When a `Conv2d` is followed immediately by `BatchNorm2d`, the conv `bias` is redundant — BN re-centers the activations afterward. Set `bias=False` on those convs.


In [ ]:
class SmallCNN(nn.Module):
    """Three Conv-BN-ReLU-Pool blocks + adaptive-avg-pool head."""
    def __init__(self, num_classes: int = 2) -> None:
        """Build the CNN with `num_classes` output logits."""
        super().__init__()
        # YOUR CODE HERE: self.features and self.head
        self.features: nn.Sequential = None
        self.head: nn.Sequential = None

        raise NotImplementedError()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Run a batch of images `x` of shape (B, 3, H, W) through the network and return class logits of shape (B, num_classes)."""
        return self.head(self.features(x))

model_p1 = SmallCNN(num_classes=2).to(device)
n_params = sum(p.numel() for p in model_p1.parameters())
print(f'Parameters: {n_params:,}')


> **Question 1.1** — Calculate the number of parameters in the model. Explain your reasoning, distinguishing between trainable and non-trainable parameters.

### Exercise 1.3 — Reusable `train_epoch` and `evaluate`

Same pattern from Notebooks 4–5. We will reuse these for Parts 1 and 2.

In [ ]:
def train_epoch(
    model: nn.Module,
    loader: DataLoader,
    opt: torch.optim.Optimizer,
    loss_fn: nn.Module,
) -> tuple[float, float]:
    """Train `model` for one epoch over `loader` and return (mean_loss, accuracy)."""
    # YOUR CODE HERE: standard training loop. Return (mean_loss, accuracy).
    raise NotImplementedError()

@torch.no_grad()
def evaluate(
    model: nn.Module,
    loader: DataLoader,
    loss_fn: nn.Module,
) -> tuple[float, float]:
    """Evaluate `model` over `loader` (no grad) and return (mean_loss, accuracy)."""
    # YOUR CODE HERE: standard eval loop. Return (mean_loss, accuracy).
    raise NotImplementedError()


### Exercise 1.4 — Train the CNN

Use Adam, `lr=1e-3`, `CrossEntropyLoss`, 5 epochs. Print per-epoch train and test accuracy.

In [ ]:
# YOUR CODE HERE
raise NotImplementedError()


> **Question 1.2** — Compare your final accuracy with the lecture's MNIST-1D result (~83% on a 2k-parameter conv vs. ~60% on an MLP with 75× more parameters). Why is binary cat/dog at 64×64 *harder* than MNIST-1D? List at least two factors visible in your data sample plot.

---
## Part 2 — Pretrained ResNet18 (transfer learning, ~25 min)

We just trained a network from scratch and reached modest accuracy. In any small-data setting, the right starting point is *not* random initialization — it is a backbone that has already learned a generic image vocabulary on ImageNet. We will:

1. Load a pretrained `resnet18` from `torchvision.models`.
2. **Freeze** the backbone (set `requires_grad = False` everywhere).
3. **Replace** the final classification layer with a fresh one for our 37 pet breeds.
4. Train only the new head.

This is the workflow you will use 90% of the time in industrial computer vision.

> **Practical callout — freeze, then maybe un-freeze.** Start by training only the head. Once it has converged, you may *optionally* un-freeze the last block(s) of the backbone and continue with a much smaller learning rate. We cover the freeze stage here; un-freezing is in the optional section.


### Exercise 2.1 — Build 224×224 loaders for the 37-breed task

Same dataset, but `target_types='category'`. Resolution must match what the pretrained model was trained at. Same ImageNet normalization. `batch_size=32` for training (224×224 is much bigger than 64×64); 64 for eval.

In [ ]:
SIZE_P2 = 224

# YOUR CODE HERE: tf_p2_train (with horizontal flip), tf_p2_eval, train_p2, test_p2,
#                 train_loader_p2, test_loader_p2
raise NotImplementedError()

print(f'Train: {len(train_p2)}   Test: {len(test_p2)}   Classes: 37')


### Exercise 2.2 — Load the pretrained ResNet18

Use `models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)`. The first call downloads ~45 MB and caches it under `~/.cache/torch/hub/checkpoints/`.

In [ ]:
# YOUR CODE HERE: instantiate `resnet` with the pretrained weights
raise NotImplementedError()

print(resnet.fc)  # the original 1000-way ImageNet classifier


### Exercise 2.3 — Freeze the backbone, replace the head

Loop over `resnet.parameters()` and set `requires_grad = False`. Then replace `resnet.fc` with a new `nn.Linear(resnet.fc.in_features, 37)` (whose parameters default to `requires_grad=True`). Move the model to `device`. Print **total** vs. **trainable** parameter counts — you should see ~11 M total but only ~19 K trainable.

In [ ]:
# YOUR CODE HERE
raise NotImplementedError()

n_total     = sum(p.numel() for p in resnet.parameters())
n_trainable = sum(p.numel() for p in resnet.parameters() if p.requires_grad)
print(f'Total params: {n_total:,}   Trainable: {n_trainable:,}')


> **Question 2.1** — Why is replacing only `resnet.fc` enough? Explain the intuition behind this choice.

### Exercise 2.4 — Train only the head

Use `Adam` over the trainable parameters only — pass `[p for p in resnet.parameters() if p.requires_grad]` to the optimizer. `lr=1e-3`, `CrossEntropyLoss`, 3 epochs.

In [ ]:
# YOUR CODE HERE
raise NotImplementedError()


> **Question 2.2** — Compare your accuracy and training time with Part 1. The pretrained model has *hundreds of times* more parameters but trained much faster — why? What does that tell you about the difference between *capacity* and *what the optimizer actually has to learn*?

> **Question 2.3** — If you had a budget of 15 minutes of GPU time and a brand-new vision dataset of ~1,000 labeled images, would you train from scratch or fine-tune? What if the dataset had 10,000,000 labeled images?

---
## Part 3 — U-Net for foreground segmentation (~40 min)

For each Pet image, the dataset also provides a per-pixel **trimap** mask:
- `1` = foreground (the pet)
- `2` = background
- `3` = boundary (uncertain pixels along the silhouette)

We will build a **U-Net** that predicts a foreground/background mask for any input image — the same encoder–decoder architecture used in medical segmentation and (more recently) the noise predictor inside diffusion models.

> **Practical callout — image and mask transforms must stay synchronized.** A random horizontal flip applied to the image must be applied *identically* to the mask — otherwise the supervision signal is shuffled. `transforms.Compose` cannot do this safely. The standard fix is to write a custom `Dataset` whose `__getitem__` runs the same random decisions on both.


### 3.1 — A joint image+mask `Dataset`

### Exercise 3.1 — `PetSegDataset`

Subclass `torch.utils.data.Dataset`. In `__init__`, store an internal `OxfordIIITPet(target_types='segmentation', download=False)`. In `__getitem__`:

1. Resize the image with `Image.BILINEAR` and the mask with `Image.NEAREST` to `(SIZE_P3, SIZE_P3)`. **Never bilinear-interpolate a label mask** — it invents fractional class IDs.
2. With probability 0.5 (when `augment=True`), apply `Image.FLIP_LEFT_RIGHT` to **both** image and mask.
3. Convert the image to a normalized tensor with `transforms.functional.to_tensor` + `Normalize(IMAGENET_MEAN, IMAGENET_STD)`.
4. Convert the mask to `torch.long` of shape `(H, W)` via `np.array(mask, dtype=np.int64)`.
5. Collapse the trimap into a binary mask: `(mask != 2).long()` — foreground (1) and boundary (3) become `1`; background (2) becomes `0`.


In [ ]:
SIZE_P3 = 128

class PetSegDataset(Dataset):
    """Wraps OxfordIIITPet to return synchronized (image_tensor, binary_mask) pairs."""
    def __init__(self, split: str, size: int = SIZE_P3, augment: bool = False) -> None:
        """Build the dataset for `split` ('trainval' or 'test'), resizing to `size`x`size` and optionally enabling random horizontal flip via `augment`."""
        self.base = OxfordIIITPet(DATA_ROOT, split=split, target_types='segmentation', download=False)
        self.size = size
        self.augment = augment
        self.norm = transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD)

    def __len__(self) -> int:
        """Return the number of samples in the dataset."""
        return len(self.base)

    def __getitem__(self, idx: int) -> tuple[torch.Tensor, torch.Tensor]:
        """Return (image, mask) for sample `idx` — image is (3, H, W) float, mask is (H, W) long with values in {0, 1}."""
        img, mask = self.base[idx]            # PIL.Image, PIL.Image
        # YOUR CODE HERE: resize both, optionally flip both, convert to tensors,
        #                 and collapse the trimap into a binary mask.
        raise NotImplementedError()

train_p3 = PetSegDataset('trainval', augment=True)
test_p3  = PetSegDataset('test')
print(f'Train: {len(train_p3)}   Test: {len(test_p3)}')

# num_workers=0 because PetSegDataset is defined in this notebook — Jupyter cells
# are not importable from spawned worker processes on macOS / Windows.
train_loader_p3 = DataLoader(train_p3, batch_size=16, shuffle=True,  num_workers=0)
test_loader_p3  = DataLoader(test_p3,  batch_size=32, shuffle=False, num_workers=0)


### Exercise 3.2 — Sanity check

Grab one sample and verify the image is `(3, 128, 128)` of float, the mask is `(128, 128)` of long, and the unique values in the mask are `{0, 1}`.

In [ ]:
# YOUR CODE HERE: print img.shape, img.dtype, mask.shape, mask.dtype, mask.unique()
raise NotImplementedError()


### 3.2 — Build the U-Net

### Exercise 3.3 — `SmallUNet`

The basic building block of the U-Net: two `Conv2d(3×3, padding=1) → BatchNorm2d → ReLU` stacks. Use `bias=False` on the convs (same reason as Part 1).

In [ ]:
class DoubleConv(nn.Module):
    """Two 3x3 conv-BN-ReLU layers in a row."""
    def __init__(self, cin: int, cout: int) -> None:
        """Build the block mapping `cin` input channels to `cout` output channels."""
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(cin, cout, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(cout), nn.ReLU(inplace=True),
            nn.Conv2d(cout, cout, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(cout), nn.ReLU(inplace=True),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Apply the two conv-BN-ReLU layers to `x` of shape (B, cin, H, W); returns (B, cout, H, W)."""
        return self.net(x)


Three encoder levels (channels `base → 2·base → 4·base`) with `MaxPool2d(2)` between them, a bottleneck (`8·base` channels), and three decoder levels that mirror the encoder. Each decoder step uses `ConvTranspose2d(stride=2)` to upsample, **concatenates** the upsampled tensor with the matching encoder output along the channel dim (`torch.cat([up, enc], dim=1)`), and feeds it through a `DoubleConv`. Final `Conv2d(1×1)` to `out_ch` channels (one per class).

The skip connections are the heart of the U-Net — without them, the decoder would have to *invent* high-resolution structure from a 16×16 bottleneck.


In [ ]:
class SmallUNet(nn.Module):
    """Tiny U-Net: 3 encoder levels, bottleneck, 3 decoder levels."""
    def __init__(self, in_ch: int = 3, out_ch: int = 2, base: int = 16) -> None:
        """Build the U-Net with `in_ch` input channels, `out_ch` output classes, and `base` channels at the top encoder level."""
        super().__init__()
        # YOUR CODE HERE: enc1, enc2, enc3, bottleneck, pool,
        #                 up3, dec3, up2, dec2, up1, dec1, out_conv
        raise NotImplementedError()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Run `x` of shape (B, in_ch, H, W) through encoder/bottleneck/decoder with skip concats; returns logits of shape (B, out_ch, H, W)."""
        # YOUR CODE HERE: encode, bottleneck, decode (with skip concats), out_conv
        raise NotImplementedError()

unet = SmallUNet().to(device)
print(f'U-Net params: {sum(p.numel() for p in unet.parameters()):,}')

# Sanity check
with torch.no_grad():
    x = torch.randn(2, 3, SIZE_P3, SIZE_P3, device=device)
    out = unet(x)
    print(f'Input {tuple(x.shape)} -> Output {tuple(out.shape)}')


> **Question 3.1** — What is the smallest spatial dimension your tensor reaches in the U-Net?

### 3.3 — Training and metrics

For segmentation we track two metrics:

- **Pixel accuracy** — fraction of pixels whose predicted class matches the ground truth. Easy to compute, but *unreliable on imbalanced masks*: a model that predicts "background everywhere" can score 70%+ pixel accuracy on Pet while learning nothing about the foreground.
- **IoU** (intersection-over-union) for the foreground class — the more honest metric. `IoU = |A ∩ B| / |A ∪ B|`.

> **Practical callout — pixel accuracy lies on imbalanced masks.** Always pair pixel accuracy with IoU (or its close cousin, Dice).


### Exercise 3.5 — `seg_train_epoch` and `seg_evaluate`

Like the classification helpers, but the target is `(B, H, W)` of long, the loss is `CrossEntropyLoss` over the channel axis (`logits` are `(B, 2, H, W)`), and `seg_evaluate` additionally returns the mean foreground IoU.

In [ ]:
def seg_train_epoch(
    model: nn.Module,
    loader: DataLoader,
    opt: torch.optim.Optimizer,
    loss_fn: nn.Module,
) -> tuple[float, float]:
    """Train the segmentation `model` for one epoch over `loader` and return (mean_loss, mean_pixel_accuracy)."""
    # YOUR CODE HERE: per-batch forward/backward; track mean loss and mean per-image
    # pixel accuracy. Return (mean_loss, mean_pix_acc).
    raise NotImplementedError()

@torch.no_grad()
def seg_evaluate(
    model: nn.Module,
    loader: DataLoader,
    loss_fn: nn.Module,
) -> tuple[float, float, float]:
    """Evaluate the segmentation `model` over `loader` and return (mean_loss, mean_pixel_accuracy, mean_foreground_IoU)."""
    # YOUR CODE HERE: track mean loss, mean pixel accuracy, and mean foreground IoU.
    # Compute IoU per image as |pred=1 AND true=1| / |pred=1 OR true=1|, then average.
    # Use .clamp_min(1) on the union to avoid 0/0 on images with no foreground.
    # Return (mean_loss, mean_pix_acc, mean_iou).
    raise NotImplementedError()


### Exercise 3.6 — Train the U-Net

Adam, `lr=1e-3`, `CrossEntropyLoss`, 10 epochs. Print per-epoch train pixel accuracy, test pixel accuracy, test IoU.

In [ ]:
# YOUR CODE HERE
raise NotImplementedError()


### 3.4 — Visualize predictions

### Exercise 3.7 — `apply_mask` helper

A one-liner that multiplies a `(3, H, W)` image by a `(H, W)` binary mask, broadcasting along the channel dim. Returns the foreground-only image.

In [ ]:
def apply_mask(img: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:
    """img: (3, H, W) float in [0,1], mask: (H, W) of 0/1 -> (3, H, W)."""
    # YOUR CODE HERE
    raise NotImplementedError()


Provided: sample 4 random test images and plot a 4-row column for each — original / image × ground truth / ground truth / prediction.

In [ ]:
unet.eval()
x, m = next(iter(test_loader_p3))
x, m = x.to(device), m.to(device)
with torch.no_grad():
    pred = unet(x).argmax(1).cpu()

# De-normalize images for display
mean_t = torch.tensor(IMAGENET_MEAN).view(3, 1, 1)
std_t  = torch.tensor(IMAGENET_STD ).view(3, 1, 1)
imgs = (x.cpu() * std_t + mean_t).clamp(0, 1)

N = 4
fig, axes = plt.subplots(4, N, figsize=(3*N, 12))
i_rand = np.random.randint(0, len(imgs), N)
for i, idx in enumerate(i_rand):
    axes[0, i].imshow(imgs[idx].permute(1, 2, 0));                                axes[0, i].set_title('Image');        axes[0, i].axis('off')
    axes[1, i].imshow(apply_mask(imgs[idx], m[idx].cpu()).permute(1, 2, 0));      axes[1, i].set_title('Masked image'); axes[1, i].axis('off')
    axes[2, i].imshow(m[idx].cpu(), cmap='gray');                                 axes[2, i].set_title('Ground truth'); axes[2, i].axis('off')
    axes[3, i].imshow(pred[idx], cmap='gray');                                    axes[3, i].set_title('Prediction');   axes[3, i].axis('off')
plt.tight_layout(); plt.show()


> **Question 3.3** — Look at the predicted masks. Where does the model fail? Are the failures concentrated at object boundaries, inside the body, or in cluttered backgrounds?

### 3.5 — Out-of-distribution probe

The U-Net has only seen 37 cat and dog breeds. Will it segment something it has never trained on? We test it on a Wikipedia photo of a **red fox** — visually close to a dog, but not in our 37 breeds.

> The URL below was checked at the time of authoring. If it 404s, swap it for any other animal photo.

In [ ]:
# A red fox: morphologically close to a dog, but not in the Pet dataset.
OOD_URL = 'https://upload.wikimedia.org/wikipedia/commons/3/30/Vulpes_vulpes_ssp_fulvus.jpg'

with urllib.request.urlopen(OOD_URL) as resp:
    ood_pil = Image.open(BytesIO(resp.read())).convert('RGB')
print(f'Original size: {ood_pil.size}')

ood_resized = ood_pil.resize((SIZE_P3, SIZE_P3), Image.BILINEAR)
ood_t = transforms.functional.to_tensor(ood_resized)            # (3, H, W) in [0,1]
ood_norm = transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD)(ood_t).unsqueeze(0).to(device)

unet.eval()
with torch.no_grad():
    ood_pred = unet(ood_norm).argmax(1)[0].cpu()

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
axes[0].imshow(ood_t.permute(1, 2, 0));                                   axes[0].set_title('Image (OOD: red fox)'); axes[0].axis('off')
axes[1].imshow(apply_mask(ood_t, ood_pred).permute(1, 2, 0));             axes[1].set_title('Masked image');         axes[1].axis('off')
axes[2].imshow(ood_pred, cmap='gray');                                    axes[2].set_title('Predicted mask');       axes[2].axis('off')
plt.tight_layout(); plt.show()


> **Question 3.4** — The fox is morphologically close to a dog. Does the U-Net produce a sensible foreground mask? What does that suggest about *what* the network has actually learned — fox features specifically, or a more general 'animal vs. background' representation?

---
## Optional extensions

Pick at most **one** during lab time; the rest are good homework.

### O.1 — Un-freeze and fine-tune the ResNet (Part 2 follow-up)

Un-freeze the last residual block of the ResNet (`resnet.layer4`) plus the new `fc`, train for 2–3 more epochs with a *small* learning rate (`1e-4`), and compare with the frozen-backbone result. Typically gains 1–3 percentage points — demonstrates that the deepest backbone features are slightly out-of-domain on Pets and benefit from adaptation.

### O.2 — Dice loss for the U-Net (Part 3 follow-up)

Implement a Dice loss from scratch (`1 − 2·|A∩B| / (|A|+|B|)`), combine it with cross-entropy as `total = ce + λ·dice`, retrain. Compare IoU.

### O.3 — Visualize a ResNet's feature maps

Register a forward hook on `resnet.layer1[0].conv1`, push a few Pet images through, plot the first 16 feature maps as a 4×4 grid. Connect to the lecture's discussion of receptive fields and channels.


---
## Summary

In this notebook you built three image models on a single dataset:

- **Part 1.** A small from-scratch CNN (~90 K params) on binary cat/dog at 64×64 — convolution as a strong inductive bias matching the symmetries of image data.
- **Part 2.** A pretrained ResNet18 (~11 M params, ~19 K trainable) on 37-breed classification at 224×224 — transfer learning, the dominant industrial pattern.
- **Part 3.** A small U-Net (~480 K params) on foreground segmentation at 128×128 — encoder/decoder with skip connections, the architectural backbone of every modern dense-prediction model and of diffusion-model noise predictors.

Lecture 8 introduces **transformers**, which replace fixed local kernels with learned content-dependent attention and have largely supplanted pure convolutional models on most vision benchmarks. The convolutional toolkit you built here remains essential: every high-performing vision transformer still uses convolutional stems, residual blocks, and normalization layers.
